# Comparing parallel universes - setpoint
If the user had their setpoint 1 degree lower, could they save money?

In [ ]:
# Imports
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import requests
from datetime import datetime, timezone, timedelta
import matplotlib.dates as mdates
from scipy.integrate import trapezoid

from model_helper_functions import get, plot_temp_power

In [ ]:
# Set up API call
url = "https://bristol.passivuk.com/optimisation"
with open("API_key.txt", "r") as file:
    API_key = file.read()

## Compare two schedules

#### Research question 1:
If the user had their setpoint 1 degree lower, could they save money?

In [ ]:
# Two schedules to compare
schedule_1_input = {
"num_zones": 1,
"zone_1_heating_daily_schedule": {"hours": [8, 10, 14, 16], "C": [21, 'frost', 21, 'frost']},
"start_datetime": "2024-01-01T07:00:00Z",
# "zone_1_temperature": 19 <- default
}

schedule_2_input = {
"num_zones": 1,
"zone_1_heating_daily_schedule": {"hours": [8, 10, 14, 16], "C": [20, 'frost', 20, 'frost']},
"start_datetime": "2024-01-01T07:00:00Z",
# "zone_1_temperature": 19 <- default
}

In [ ]:
return_1 = requests.post(url, headers={"X-API-Key": API_key}, json=schedule_1_input).json()
return_2 = requests.post(url, headers={"X-API-Key": API_key}, json=schedule_2_input).json()

### Plot results

In [ ]:
output_data = []
for outputs in [return_1, return_2]:
    if outputs['success']:
        dts = [datetime.fromisoformat(t.replace("Z", "+00:00")) for t in outputs["real_dt"]]
        n = len(dts)
        S = outputs["State"]
        df = pd.DataFrame({
            "Datetimes": dts,
            "Setpoint Z1": get(S, n, "setpoint.z1"),
            "Room Temp Z1": get(S, n, "room_temp.z1"),
            "External Temp": get(S, n, "ext"),
            "Input Power": get(S, n, "E.hs1.heat.z1"),
            "Output Power": get(S, n, "U.hs1.z1"),
            "Elec Cost (p/kWh)": get(S, n, "elec_cost"),
            "Tank Temp": get(S, n, "tank_temp"),
            "HW Setpoint": get(S, n, "setpoint_hw")
            # "Tariff":get(S, n, "tariff")
        })
        output_data.append(df)
        plot_temp_power(df, schedule_1_input.get("num_zones"))

### Calculate energy costs

In [ ]:
def get_time_passed(df, start_time=datetime(year=2024, month=1, day=1, hour=0)):
    time_in_hours = []
    start_time = start_time.replace(tzinfo=timezone(timedelta(hours=0, minutes=0)))
    for time in df.Datetimes:
        time_elapsed = time-start_time  # Time deltas store time in days and seconds (for reasons....)
        time_in_hours.append(time_elapsed.days*24 + time_elapsed.seconds/(60*60))  # Time in hours since the start!
    return time_in_hours

In [ ]:
for df in output_data:
        power_df = df.copy(deep=True).dropna()
        time_in_hours = get_time_passed(power_df)
        if len(set(power_df["Elec Cost (p/kWh)"])) == 1:
                energy_used = trapezoid(power_df["Input Power"], x=time_in_hours)
                print("Total energy used: ", round(energy_used, 2), " kWh")
                print("Energy cost: £", round(energy_used*power_df["Elec Cost (p/kWh)"][1]/100, 2))
        
        # else:
        #       need to go through dataframe
        #       integrate curve between times where energy cost values change
        #       calculate energy for each tariff + therefore cost for each chunk
        #       sum up to get total


In [ ]:
S1 = return_1["State"]
S2 = return_2["State"]
combined_df = pd.DataFrame({
    "Datetimes": [datetime.fromisoformat(t.replace("Z", "+00:00")) for t in return_1["real_dt"]],
    "S21 Setpoint": get(S1, n, "setpoint.z1"),
    "S21 Room Temp": get(S1, n, "room_temp.z1"),
    "S20 Setpoint": get(S2, n, "setpoint.z1"),
    "S20 Room Temp": get(S2, n, "room_temp.z1"),
    "External Temp": get(S1, n, "ext"),
    "S21 Input Power": get(S1, n, "E.hs1.heat.z1"),
    "S20 Input Power": get(S2, n, "E.hs1.heat.z1"),
    "Elec Cost (p/kWh)": get(S1, n, "elec_cost"),
    
})

In [ ]:
matplotlib.rcParams.update({'font.size': 16})

In [ ]:
def plot_comparison(df, axis_int=6, share_x=True, plot_tariff=True):
    """Plot comparison of temperature, power, and cost from the model calculations
    
    Args:
        df (DataFrame): model output data
        axis_int (int): interval for the x axis (defaults to 6 hours)
        share_x (bool): whether or not to have subfigures share x axis
        plot_tariff (bool): whether or not to plot tariff information
    """

    # --- Set up ---
    dts = df["Datetimes"]
    if plot_tariff:
        fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=share_x)
    else:
        fig, axes = plt.subplots(2, 1, figsize=(15, 6.5), sharex=share_x)

    # --- Panel 1: Temperatures ---
    ax = axes[0]
    # Scenario 1
    ax.plot(dts, df["S21 Room Temp"], label="S21 Room", color="#dc267f",    linewidth=2.5)
    ax.plot(dts, df["S21 Setpoint"],  label="S21 Setpoint",  color="#dc267f",    linewidth=2, linestyle="-.")

    # Scenario 2
    ax.plot(dts, df["S20 Room Temp"], label="S20 Room", color="#ffb000", linewidth=2.5)
    ax.plot(dts, df["S20 Setpoint"],  label="S20 Setpoint",  color="#ffb000",    linewidth=2, linestyle="--")
    
    # External temperature
    ax.plot(dts, df["External Temp"], label="External", color="#631ff3", linewidth=2, linestyle="--")
    ax.set_ylabel("Temperature (°C)")
    ax.legend(bbox_to_anchor=(1.0, 1.0))
    ax.grid(True, alpha=0.3)

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=axis_int))

    # --- Panel 2: Heat pump power ---
    ax = axes[1]
    ax.plot(dts, df["S21 Input Power"], label="S21 Power Usage",  color="#1589e8", linewidth=2.5)
    ax.plot(dts, df["S20 Input Power"], label="S20 Power Usage", color="#1589e8", linewidth=2.5, linestyle="--")
    
    ax.set_ylabel("Power (kW)")
    ax.legend(bbox_to_anchor=(1.27, 1.0))
    ax.grid(True, alpha=0.3)


    ax.xaxis.set_major_locator(mdates.HourLocator(interval=axis_int))
    if not plot_tariff:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %H:%M"))
        plt.xticks(rotation=30, ha="right")
    else:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    # --- Panel 3: Hot water tank + electricity cost ---
    if plot_tariff:
        ax = axes[2]
        ax.grid(True, alpha=0.3)

        # ax2 = ax.twinx()
        ax.plot(dts, df["Elec Cost (p/kWh)"], label="Elec Cost (p/kWh)", color="#fe5100", linewidth=2.5, linestyle="-.")
        ax.set_ylabel("Elec Cost (p/kWh)")
        ax.tick_params(axis="y")

        ax.legend(bbox_to_anchor=(1.0, 1.0))

        # X axis formatting
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %H:%M"))
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=axis_int))
        plt.xticks(rotation=30, ha="right")

    plt.tight_layout()
    # plt.savefig("/Users/mm25873/Desktop/passiv/graphs_output/passiv_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    # print("Plot saved to passiv_results.png")

    duration_hours = (list(dts)[-1] - list(dts)[0]).total_seconds() / 3600
    print(f"Data covers {duration_hours:.1f} hours ({len(dts)} timesteps)")


In [ ]:
plot_comparison(combined_df, plot_tariff=False)